# V1 Pipeline

End-to-end run for one PDF:

```
mineru extraction -> pdf_origin_middle.json -> postprocess ->
classify images (semantic/decorative) -> caption semantic images ->
unify text
```

Unlike the dual-pipeline track (`dual_pipeline_parsing.ipynb` +
`image_understanding.ipynb`, which transcribes each page with Qwen-VL),
this pipeline gets page text from MinerU's own layout tree via
`post_process.pipeline.process_mineru_json` (`_middle.json` ->
section-aware paragraphs). `post_process.layout.flatten` drops image
blocks by design (text-only), so visuals are extracted separately from
`_middle.json` via `ingestion.middle_visuals.extract_visuals` and
joined back to postprocessed pages by page number.

## Setup

In [ ]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent  # notebook lives in notebooks/
sys.path.insert(0, str(PROJECT_ROOT / "src"))

MINERU_BIN = PROJECT_ROOT / ".venv-mineru" / "bin" / "mineru"
assert MINERU_BIN.exists(), (
    f"MinerU venv not found at {MINERU_BIN}. Create it with:\n"
    f"  uv venv .venv-mineru --python 3.13\n"
    f'  uv pip install --python .venv-mineru "mineru[core]"'
)

PDF_PATH = PROJECT_ROOT / "notebooks" / "Anatomy of Face and Oral Cavity - Basic of DEMN.pdf_origin.pdf"
OUTPUT_ROOT = PROJECT_ROOT / "output"
assert PDF_PATH.exists(), PDF_PATH

PDF_STEM = PDF_PATH.stem  # MinerU strips the .pdf extension for its run dir name
RUN_DIR = OUTPUT_ROOT / PDF_STEM / "auto"  # "auto" = pipeline backend's dir suffix

from post_process.pipeline import process_mineru_json
from ingestion.middle_visuals import extract_visuals
from ingestion.unify import build_unified_items_from_postprocessed, render_unified_text, render_unified_markdown
from ingestion.semantic_chunk import SemanticChunker
from captioning.qwen_vl import QwenVLCaptioner

PROJECT_ROOT, PDF_PATH

/Users/michaeleko/Documents/Works/aiml-institute/challenge-2/intelligent-tutoring-system-for-medical-student/.venv-mineru/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(PosixPath('/Users/michaeleko/Documents/Works/aiml-institute/challenge-2/intelligent-tutoring-system-for-medical-student'),
 PosixPath('/Users/michaeleko/Documents/Works/aiml-institute/challenge-2/intelligent-tutoring-system-for-medical-student/notebooks/Anatomy of Face and Oral Cavity - Basic of DEMN.pdf_origin.pdf'))

## MinerU extraction

Runs MinerU (pipeline backend, `.venv-mineru`) on the source PDF to
produce `{PDF_STEM}_middle.json` under `RUN_DIR`.

In [2]:
import subprocess

cmd = [
    str(MINERU_BIN),
    "-p", str(PDF_PATH),
    "-o", str(OUTPUT_ROOT),
    "-b", "pipeline",
    "-m", "auto",
]

result = subprocess.run(cmd, cwd=PROJECT_ROOT, capture_output=True, text=True)
print(result.stdout[-4000:])
if result.returncode != 0:
    print(result.stderr[-4000:])
result.returncode

Start MinerU FastAPI Service: http://127.0.0.1:50966
API documentation: http://127.0.0.1:50966/docs



0

In [3]:
MIDDLE_PATH = RUN_DIR / f"{PDF_STEM}_middle.json"

with open(MIDDLE_PATH) as f:
    middle_document = json.load(f)

len(middle_document["pdf_info"])

68

## Postprocess

8-stage pipeline (`post_process.pipeline.process_mineru_json`): parse
-> flatten -> feature extraction -> header/footer removal ->
classification -> reading order -> semantic tree -> JSON render.
Produces section-aware paragraphs per page; image blocks are dropped
here (handled separately below).

In [4]:
postprocessed = process_mineru_json(str(MIDDLE_PATH))

postprocessed_path = RUN_DIR / f"{PDF_STEM}_post_processed.json"
with open(postprocessed_path, "w") as f:
    json.dump(postprocessed, f, indent=2, ensure_ascii=False)

postprocessed["total_pages"], postprocessed["pages"][0]

(68,
 {'page_num': 1,
  'section': '',
  'title': 'THE DIGESTIVE SYSTEM: FACE AND ORAL CAVITY',
  'text': 'Basic of DEMN System',
  'nodes': [{'id': 'para_p0_b2',
    'type': 'PARAGRAPH',
    'text': 'Basic of DEMN System',
    'page': 1,
    'bbox': [520, 371, 751, 397]}]})

## Extract visuals from `_middle.json`

Pulls image/chart/table-as-image crops directly from the MinerU
middle document (grouped by page_idx), since `post_process` drops
them. A `table` only lands here if MinerU couldn't OCR it to HTML
(scanned/image table) — HTML tables already went through
`post_process` as text.

In [5]:
visuals_by_page_idx = extract_visuals(middle_document)

total_visuals = sum(len(v) for v in visuals_by_page_idx.values())
total_visuals, next(iter(visuals_by_page_idx.values()))[0]

(117,
 MiddleVisual(item_id='p0_b1', page_idx=0, type='image', bbox=[0, 0, 477, 538], img_path='95c415eff3c7336c1bae149573981b0a4993e481367ea78323e2955cd02bb7f9.jpg', relevance=None))

## Classify visual relevance (semantic vs. decorative)

For each visual, ask Qwen-VL whether it supports the page's
postprocessed text or is purely decorative, using that page's text as
context (same `classify_image_relevance` used in the dual-pipeline
track).

In [6]:
from tqdm.auto import tqdm

page_text_by_idx = {p["page_num"] - 1: p["text"] for p in postprocessed["pages"]}

captioner = QwenVLCaptioner()

for page_idx, visuals in tqdm(visuals_by_page_idx.items(), desc="Classifying visuals"):
    slide_text = page_text_by_idx.get(page_idx, "")
    for visual in visuals:
        img_path = RUN_DIR / "images" / visual.img_path
        visual.relevance = captioner.classify_image_relevance(str(img_path), slide_text)

sum(v.relevance == "semantic" for visuals in visuals_by_page_idx.values() for v in visuals)

Classifying visuals:   0%|          | 0/60 [00:00<?, ?it/s]The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 5/5 [00:00<00:00, 83.60it/s]
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Classifying visuals: 100%|██████████| 60/60 [04:24<00:00,  4.41s/it]


79

## Caption semantic visuals only

Decorative visuals are skipped entirely — no caption generated, and
they won't appear in the unified document.

In [7]:
semantic_visuals = [
    v for visuals in visuals_by_page_idx.values() for v in visuals
    if v.relevance == "semantic"
]

caption_results = []
for visual in tqdm(semantic_visuals, desc="Captioning visuals"):
    img_path = RUN_DIR / "images" / visual.img_path
    caption = captioner.caption(str(img_path))
    caption_results.append({
        "item_id": visual.item_id,
        "image_path": str(img_path),
        "content_type": visual.type,
        "caption": caption,
    })

captioner.unload()

captions_path = RUN_DIR / f"{PDF_STEM}_v1_captions.json"
with open(captions_path, "w") as f:
    json.dump(caption_results, f, indent=2)

len(caption_results)

Captioning visuals: 100%|██████████| 79/79 [1:34:28<00:00, 71.75s/it]   


79

## Unify text

Merges postprocessed page text with inlined `[FIGURE:item_id]`
caption markers for semantic visuals, in page order.

In [8]:
captions = {r["item_id"]: r["caption"] for r in caption_results}
unified_items = build_unified_items_from_postprocessed(postprocessed, visuals_by_page_idx, captions)
unified_text = render_unified_text(unified_items)

print(f"{len(unified_items)} unified items, {len(unified_text)} chars")
print(unified_text[:1500])

140 unified items, 142410 chars
Basic of DEMN System

<sup>▪</sup> Digestive System: The system whose function it is to break down foods into molecules small enough to enter body cells.

<sup>▪</sup> Allows the body to ingest & digest proteins, fats & carbohydrates & absorb them into the bloodstream & lymph to be taken to body cells for metabolism & conversion to ATP.

<sup>▪</sup> Gastrointestinal (GI) Tract aka Alimentary Canal: A continuous tube that extends from the mouth to the anus.

<sup>▪</sup> Tonus: The sustained muscular contraction of the GI tract walls that helps to move food along.

[FIGURE:p2_b2] The image is a detailed anatomical illustration of the digestive system. It shows a human figure from the side, with the internal organs labeled. Here's a breakdown of the visual elements:

1. **Top Left Region:**
   - The text "stive System" is written at the top left corner.

2. **Mouth (oral cavity):** 
   - Located at the top of the head, pointing to the mouth area.
   
3. *

In [9]:
unified_text_path = RUN_DIR / f"{PDF_STEM}_v1_unified_text.txt"
with open(unified_text_path, "w") as f:
    f.write(unified_text)

unified_markdown = render_unified_markdown(unified_items)
unified_md_path = RUN_DIR / f"{PDF_STEM}_v1_unified_text.md"
with open(unified_md_path, "w") as f:
    f.write(unified_markdown)

unified_text_path, unified_md_path

(PosixPath('/Users/michaeleko/Documents/Works/aiml-institute/challenge-2/intelligent-tutoring-system-for-medical-student/output/Anatomy of Face and Oral Cavity - Basic of DEMN.pdf_origin/auto/Anatomy of Face and Oral Cavity - Basic of DEMN.pdf_origin_v1_unified_text.txt'),
 PosixPath('/Users/michaeleko/Documents/Works/aiml-institute/challenge-2/intelligent-tutoring-system-for-medical-student/output/Anatomy of Face and Oral Cavity - Basic of DEMN.pdf_origin/auto/Anatomy of Face and Oral Cavity - Basic of DEMN.pdf_origin_v1_unified_text.md'))

## Rebuild the unified document

Lets chunking run standalone from saved artifacts instead of relying
on in-kernel state from the cells above: reloads `_middle.json` (for
`extract_visuals`), `_post_processed.json`, and `_v1_captions.json`
from disk and rebuilds the `UnifiedItem` list via
`build_unified_items_from_postprocessed`.

In [5]:
MIDDLE_PATH = RUN_DIR / f"{PDF_STEM}_middle.json"
postprocessed_path = RUN_DIR / f"{PDF_STEM}_post_processed.json"
captions_path = RUN_DIR / f"{PDF_STEM}_v1_captions.json"

with open(MIDDLE_PATH) as f:
    middle_document = json.load(f)

with open(postprocessed_path) as f:
    postprocessed = json.load(f)

with open(captions_path) as f:
    caption_results = json.load(f)

captions = {r["item_id"]: r["caption"] for r in caption_results}

visuals_by_page_idx = extract_visuals(middle_document)
for visuals in visuals_by_page_idx.values():
    for visual in visuals:
        # relevance classification itself isn't persisted — only captioned
        # (i.e. semantic) visuals were saved to caption_results, so presence
        # of a caption is what "semantic" meant by the time this ran.
        visual.relevance = "semantic" if visual.item_id in captions else "decorative"

unified_items = build_unified_items_from_postprocessed(postprocessed, visuals_by_page_idx, captions)

len(unified_items), unified_items[0]

(140,
 UnifiedItem(item_id='page0#text', page_idx=0, content_type='text', text='Basic of DEMN System', text_level=None))

## Semantic chunking

Groups `unified_items` into topically coherent chunks by embedding
similarity (`all-MiniLM-L6-v2`): a boundary is placed wherever
consecutive-item cosine distance exceeds a percentile-based threshold,
with a `max_chars` fallback split for long on-topic runs. Feeds
downstream concept extraction.

In [6]:
chunker = SemanticChunker()
chunks = chunker.chunk(unified_items, percentile=95.0, max_chars=6000)
chunker.unload()

len(chunks), [len(c) for c in chunks][:20]

(30, [1, 5, 5, 1, 8, 6, 6, 4, 5, 6, 6, 4, 5, 5, 3, 4, 5, 4, 3, 5])

## Inspect a sample chunk

In [7]:
sample_chunk = chunks[len(chunks) // 2]
print(f"{len(sample_chunk)} items, item_ids: {[i.item_id for i in sample_chunk]}\n")
print(render_unified_markdown(sample_chunk))

4 items, item_ids: ['p33_b23', 'page34#text', 'p34_b8', 'page35#text']

> [FIGURE:p33_b23] The image is a detailed anatomical illustration of the pharynx and related structures. Here's a breakdown of the visual elements:

1. **Tensor veli palatini**: This muscle is labeled at the top left of the image, pointing to a triangular-shaped structure near the auditory tube.
2. **Auditory tube**: Located just below the tensor veli palatini, this structure is depicted as a small, curved tube.
3. **Levator veli palatini**: This muscle is shown on the right side of the image, pointing to a triangular area above the soft palate.
4. **Salpingopharyngeus**: This muscle is labeled in the middle of the image, pointing to a band-like structure that runs horizontally across the pharynx.
5. **Superior constrictor**: This muscle is labeled below the salpingopharyngeus, pointing to a band-like structure that appears to be part of the pharyngeal wall.
6. **Soft palate**: This structure is labeled below the 

## Save chunks

Each chunk saved as its rendered markdown plus the item_ids it spans —
the item_ids are what a later concept-extraction stage would attach to
extracted concepts for content linkage back to source.

In [8]:
import re

HEADING_RE = re.compile(r"^#{1,6}\s+(.+)$", re.MULTILINE)


def first_heading(text: str) -> str | None:
    match = HEADING_RE.search(text)
    return match.group(1).strip() if match else None


doc_title = first_heading(unified_items[0].text) or PDF_STEM

stage1_chunks = []
stage1_figures = []
char_cursor = 0

for i, chunk in enumerate(chunks):
    chunk_id = f"c{i:03d}"
    chunk_text = render_unified_markdown(chunk)
    section_title = first_heading(chunk_text)
    section_path = [doc_title, section_title] if section_title else [doc_title]

    char_start = char_cursor
    char_end = char_start + len(chunk_text)
    char_cursor = char_end

    stage1_chunks.append({
        "chunk_id": chunk_id,
        "section_path": section_path,
        "char_start": char_start,
        "char_end": char_end,
        "text": chunk_text,
    })

    for item in chunk:
        if item.content_type not in ("image", "chart", "table", "diagram"):
            continue
        caption = captions.get(item.item_id)
        if not caption:
            continue
        stage1_figures.append({
            "fig_id": item.item_id,
            "image_path": next(
                (r["image_path"] for r in caption_results if r["item_id"] == item.item_id),
                None,
            ),
            "caption": caption,
            "referenced_from": chunk_id,
        })

stage1_output = {
    "doc_id": PDF_STEM,
    "source": {
        "source_id": PDF_STEM,
        "title": doc_title,
    },
    "chunks": stage1_chunks,
    "figures": stage1_figures,
}

chunks_path = RUN_DIR / f"{PDF_STEM}_v1_chunks.json"
with open(chunks_path, "w") as f:
    json.dump(stage1_output, f, indent=2)

chunks_path

PosixPath('/Users/michaeleko/Documents/Works/aiml-institute/challenge-2/intelligent-tutoring-system-for-medical-student/output/Anatomy of Face and Oral Cavity - Basic of DEMN.pdf_origin/auto/Anatomy of Face and Oral Cavity - Basic of DEMN.pdf_origin_v1_chunks.json')